# 도형 검출기(cube.pt) Colab 재학습 — self-training 루프

흰 다면체 4종(cube / octahedron / dodecahedron / icosahedron) YOLOv8n 검출기 학습.

**매 라운드 정석:** 누적·교정된 전체 데이터로 `yolov8n.pt`(COCO 사전학습)에서 **새로** 학습 →
`best.pt`로 젯슨 `models/cube.pt` 교체. (cube.pt에서 이어서 학습 ❌ — 교정 의미 반감 + 과적합)

**주의사항**
- 업로드할 zip: **교정 후** 데이터셋 (예: `dataset_labeled_2026-06-24.zip`). `dataset.zip`(교정 전) 쓰지 말 것.
- zip은 라벨 안 한 새 캡처가 빠진 상태여야 함 (라벨 없는 이미지는 학습 제외됨).
- `data.yaml`의 절대경로/`train==val` 문제는 [셀2]가 코랩용으로 새로 만들어 해결함.
- 런타임 → GPU 켜고 실행.

In [ ]:
# [셀1] 업로드 + 압축해제  — 교정 후 zip 선택 (dataset_labeled_*.zip)
from google.colab import files
up = files.upload()
import os
ZIP = next(iter(up))            # 업로드한 파일명 자동 사용
print('uploaded:', ZIP)
!unzip -o -q "$ZIP" -d /content

In [ ]:
# [셀2] train/val 분리 + 코랩용 data.yaml 생성
import os, glob, random, shutil, yaml
random.seed(0)
ROOT = '/content/dataset'
HELD_OUT_VAL = True      # ← False면 train==val (전체 학습, 검증은 live로)
VAL_FRAC = 0.15
lbl = lambda p: f"{ROOT}/labels/" + os.path.splitext(os.path.basename(p))[0] + ".txt"
pairs = [(i, lbl(i)) for i in sorted(glob.glob(f'{ROOT}/images/*')) if os.path.exists(lbl(i))]
print('pairs:', len(pairs))
if HELD_OUT_VAL:
    random.shuffle(pairs); n = int(len(pairs) * VAL_FRAC)
    for split, items in [('train', pairs[n:]), ('val', pairs[:n])]:
        for s in ('images', 'labels'):
            os.makedirs(f'{ROOT}/{split}/{s}', exist_ok=True)
        for img, lb in items:
            shutil.copy(img, f'{ROOT}/{split}/images/'); shutil.copy(lb, f'{ROOT}/{split}/labels/')
    print('train', len(pairs) - n, '/ val', n)
    data = dict(path=ROOT, train='train/images', val='val/images')
else:
    data = dict(path=ROOT, train='images', val='images')
data.update(nc=4, names=['cube', 'octahedron', 'dodecahedron', 'icosahedron'])
yaml.safe_dump(data, open(f'{ROOT}/data_colab.yaml', 'w'))
print(open(f'{ROOT}/data_colab.yaml').read())

In [ ]:
# [셀3] 학습 (기존 cube.pt와 동일 아키텍처 yolov8n, 사전학습 백본에서 새로 학습)
!pip -q install ultralytics
from ultralytics import YOLO
YOLO('yolov8n.pt').train(data='/content/dataset/data_colab.yaml',
    epochs=100, imgsz=640, batch=16, patience=30, name='cube_v2')

In [ ]:
# [셀4] best.pt 내려받기 → 젯슨 models/cube.pt 교체
from google.colab import files
files.download('runs/detect/cube_v2/weights/best.pt')